# Day 3 動手練習：預訓練資料清理管線（Data Cleaning Pipeline）

搭配 [day03.md](./day03.md) 裡的這一句：

> 訓練資料也從書籍擴大到網路世界，也就是所謂的 WebText：研究團隊蒐集 Reddit 上至少獲得 3 個 karma 的外部連結，**經過清理與去除重複後**，形成超過 800 萬份文件、約 40 GB 的文字資料。

「清理與去除重複」聽起來只是一句話帶過，但實際上是一整條管線。這份 notebook 動手做一個縮小版，模擬 GPT-2 / LLaMA 這類模型在餵資料之前，會怎麼處理從網路上抓下來的「髒資料」：

1. **HTML 清除 + 語言過濾**：網頁下載下來還帶著標籤，也可能混雜其他語言
2. **MinHash 去重**：完全重複、或幾乎重複的文章只留一份
3. **PII 遮蔽**：把 Email、信用卡卡號、電話等個資蓋掉
4. **重複片語清除**：把洗版式的重複字句壓縮成一次


## 環境安裝

這次只需要 `datasketch`（MinHash 去重）、`beautifulsoup4`（HTML 清除）、`pandas`。不需要下載模型、不需要 API key。

In [1]:
%pip install -q datasketch beautifulsoup4 pandas

Note: you may need to restart the kernel to use updated packages.


## 準備一批「髒」資料

模擬從網路上蒐集到、還沒清理過的 10 篇短文。裡面刻意混入幾種常見的「髒資料」：完全重複、幾乎重複（只改一兩個字）、還包著 HTML 標籤、夾帶個資、混入非中文內容，以及一篇乾淨到可以直接留下來的正常段落。

In [2]:
import pandas as pd

raw_dataset = [
    # 1. 個資 + HTML 連結標籤
    "您好！如需聯絡，請寄信到 support@ai-news.tw 或撥打 0912-345-678。"
    "您的信用卡卡號 4111111111111111 已經過期。本訊息僅提供收件者參考，"
    "詳見 <a href='https://fake-site.tw'>本站</a>。",

    # 2. 非中文內容（日文）
    "これは日本語で書かれた記事です。人工知能の発展について紹介します。連絡先：03-1234-5678",

    # 3. 包著 HTML 標籤的新聞
    "<html><body><div><h1>頭條快訊</h1><p>OpenAI 宣布推出新一代語言模型！</p></div>"
    "<footer>版權所有，轉載請註明出處</footer></body></html>",

    # 4. 洗版式重複片語（廣告文）
    "限時優惠不要錯過！限時優惠不要錯過！限時優惠不要錯過！立即購買 AI 課程！",

    # 5. 正常文章 A
    "GPT-3 擁有 1,750 億個參數，透過 Few-shot 就能完成新任務，詳情請見官方部落格。",

    # 6. 跟第 5 篇幾乎一樣，只改了最後幾個字（近似重複）
    "GPT-3 擁有 1,750 億個參數，透過 Few-shot 就能完成新任務，詳情請見官方文件。",

    # 7. 個資（Email + 信用卡）
    "歡迎加入我們的 AI 讀書會，報名請寄 join_ai@example.com 或私訊，銀行卡號 378282246310005 僅供測試。",

    # 8. 非中文內容（英文）
    "Large Language Models like GPT-3 are transforming how we build AI applications with few-shot prompting.",

    # 9. 乾淨的正常段落，應該要撐過整條管線活到最後
    "根據 GPT-3 論文的實驗結果，模型規模越大，Few-shot 表現通常也越好，但這不代表模型完全理解了任務本身。",

    # 10. 跟第 4 篇完全一模一樣（完全重複）
    "限時優惠不要錯過！限時優惠不要錯過！限時優惠不要錯過！立即購買 AI 課程！",
]

df = pd.DataFrame({"Raw Text": raw_dataset})
print(f"原始資料筆數：{len(df)}")
for i, text in enumerate(df["Raw Text"], 1):
    print(f"{i:>2}. {text[:40]}{'...' if len(text) > 40 else ''}")

原始資料筆數：10
 1. 您好！如需聯絡，請寄信到 support@ai-news.tw 或撥打 0912...
 2. これは日本語で書かれた記事です。人工知能の発展について紹介します。連絡先：03-...
 3. <html><body><div><h1>頭條快訊</h1><p>OpenAI ...
 4. 限時優惠不要錯過！限時優惠不要錯過！限時優惠不要錯過！立即購買 AI 課程！
 5. GPT-3 擁有 1,750 億個參數，透過 Few-shot 就能完成新任務，...
 6. GPT-3 擁有 1,750 億個參數，透過 Few-shot 就能完成新任務，...
 7. 歡迎加入我們的 AI 讀書會，報名請寄 join_ai@example.com ...
 8. Large Language Models like GPT-3 are tra...
 9. 根據 GPT-3 論文的實驗結果，模型規模越大，Few-shot 表現通常也越好...
10. 限時優惠不要錯過！限時優惠不要錯過！限時優惠不要錯過！立即購買 AI 課程！


---
## Step 1｜HTML 清除 + 語言過濾

先用 `BeautifulSoup` 把 HTML 標籤剝掉，再判斷是不是我們要的語言（這裡設定只留中文）。

**踩到的坑**：原始教材用 `langdetect` 這個套件判斷語言，但實測發現它對短篇的**繁體中文**很不準——常常把中文段落誤判成韓文。這是真的會發生的資料清理陷阱，剛好呼應「garbage in, garbage out」：清理工具本身也可能是髒資料的來源。

這裡改用一個更陽春、但透明好懂的規則：數一下文字裡漢字、假名（日文特有）、拉丁字母各佔多少比例，用比例來判斷語言。這不是嚴謹的語言學方法，只適合教學示範；真正要上線用，還是建議用 `fasttext` 的語言辨識模型或更成熟的工具。

In [3]:
import re
from bs4 import BeautifulSoup


def detect_zh_or_other(text):
    han = len(re.findall(r"[\u4e00-\u9fff]", text))
    kana = len(re.findall(r"[\u3040-\u30ff]", text))
    latin = len(re.findall(r"[A-Za-z]", text))
    total = han + kana + latin

    if total == 0:
        return "unknown"
    if kana / total > 0.05:   # 出現假名，判定為日文
        return "ja"
    if han / total >= 0.5:
        return "zh"
    if latin / total >= 0.5:
        return "en"
    return "mixed"            # 中英文比例接近，無法明確判斷


def clean_html_and_filter_lang(texts, lang="zh"):
    kept = []
    for text in texts:
        stripped = BeautifulSoup(text, "html.parser").get_text(separator=" ")
        stripped = re.sub(r"\s+", " ", stripped).strip()
        stripped = re.sub(r"\s+([，。！？、])", r"\1", stripped)
        detected = detect_zh_or_other(stripped)
        if detected == lang:
            kept.append(stripped)
        else:
            print(f"  ❌ 剔除（判定為 {detected}）：{stripped[:25]}...")
    return kept


step1 = clean_html_and_filter_lang(df["Raw Text"].tolist())
print(f"\n清理前：{len(df)} 筆 → 清理後：{len(step1)} 筆")

  ❌ 剔除（判定為 ja）：これは日本語で書かれた記事です。人工知能の発展につ...
  ❌ 剔除（判定為 en）：Large Language Models lik...

清理前：10 筆 → 清理後：8 筆


---
## Step 2｜MinHash 去重

**核心觀念速記**

- **Jaccard 相似度**：衡量兩篇文章的重疊程度，交集除以聯集。
- **MinHash**：把一篇文章「壓縮」成固定大小的簽名（這裡用 128 個雜湊值），簽名之間的相似度可以逼近原本的 Jaccard 相似度，不用整篇逐字比對。
- **LSH（局部敏感雜湊）**：建立一個「神奇抽屜櫃」，相似度高的文章會被分到同一格，比對時只要查同一格的鄰居就好，不必跟資料庫裡每一篇都比一次。


In [4]:
from datasketch import MinHash, MinHashLSH


def get_char_shingles(text, k=2):
    text = text.replace(" ", "")
    if len(text) < k:
        return {text} if text else set()
    return set(text[i:i + k] for i in range(len(text) - k + 1))


def minhash_deduplication(texts, threshold=0.7, num_perm=128):
    lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)
    unique_texts = []
    for i, doc in enumerate(texts):
        m = MinHash(num_perm=num_perm)
        for shingle in get_char_shingles(doc):
            m.update(shingle.encode("utf8"))
        if lsh.query(m):
            print(f"  🗑️ 判定為重複，已略過：{doc[:25]}...")
        else:
            lsh.insert(f"doc{i}", m)
            unique_texts.append(doc)
    return unique_texts


step2 = minhash_deduplication(step1)
print(f"\n去重前：{len(step1)} 筆 → 去重後：{len(step2)} 筆")

  🗑️ 判定為重複，已略過：GPT-3 擁有 1,750 億個參數，透過 Fe...
  🗑️ 判定為重複，已略過：限時優惠不要錯過！限時優惠不要錯過！限時優惠不要錯...

去重前：8 筆 → 去重後：6 筆


**需要留意的地方**（教學示範的簡化，不是真正生產環境會遇到的全部細節）：

- 太短的文字切出來的 shingle 集合很小甚至是空的，相似度計算容易不穩定。
- 字元級 shingle 會丟掉「詞序以外」的重複資訊，兩篇用詞完全不同、但字元恰好重疊很多的文章，理論上也可能被誤判為重複。
- 標點符號、換行等格式差異都會影響 shingle，實務上通常會先做一輪正規化（統一標點、去除多餘空白）再算 MinHash。
- 兩篇被判定重複時，程式只會留下先出現的那一篇——留下哪一篇是「誰先進來」決定的，不是「誰品質比較好」決定的。

---
## Step 3｜PII 遮蔽

用正規表示式把 Email、信用卡卡號、電話號碼蓋掉。電話號碼的格式改成台灣手機門號（`09xx-xxx-xxx`），信用卡與 Email 的規則維持原本教材的寫法。

In [5]:
def strip_pii(text):
    text = re.sub(r"[\w.-]+@[\w.-]+", "[EMAIL]", text)
    text = re.sub(r"\b\d{13,19}\b", "[CREDIT_CARD]", text)
    text = re.sub(r"09\d{2}[-\s]?\d{3}[-\s]?\d{3}", "[PHONE]", text)
    return text


step3 = [strip_pii(t) for t in step2]
for t in step3:
    print(" -", t)

 - 您好！如需聯絡，請寄信到 [EMAIL] 或撥打 [PHONE]。您的信用卡卡號 [CREDIT_CARD] 已經過期。本訊息僅提供收件者參考，詳見 本站。
 - 頭條快訊 OpenAI 宣布推出新一代語言模型！ 版權所有，轉載請註明出處
 - 限時優惠不要錯過！限時優惠不要錯過！限時優惠不要錯過！立即購買 AI 課程！
 - GPT-3 擁有 1,750 億個參數，透過 Few-shot 就能完成新任務，詳情請見官方部落格。
 - 歡迎加入我們的 AI 讀書會，報名請寄 [EMAIL] 或私訊，銀行卡號 [CREDIT_CARD] 僅供測試。
 - 根據 GPT-3 論文的實驗結果，模型規模越大，Few-shot 表現通常也越好，但這不代表模型完全理解了任務本身。


---
## Step 4｜重複片語清除

有些內容不是整篇重複，而是同一句話在同一篇裡面反覆洗版（像廣告文案）。原始教材是照英文的空白斷詞去抓 n-gram，一樣不適用中文，這裡改成依中文標點（，。！？、）切成短句，同一句話出現 3 次以上就只保留一次。

In [6]:
from collections import Counter


def remove_repetitive_phrases(text, min_len=4, threshold=3):
    clauses = re.split(r"[，。！？、\s]+", text)
    clauses = [c for c in clauses if len(c) >= min_len]

    counts = Counter(clauses)
    repetitive = {c for c, n in counts.items() if n >= threshold}

    for phrase in repetitive:
        escaped = re.escape(phrase)
        pattern = rf"(?:{escaped}[，。！？、]*\s*){{{threshold},}}"
        text = re.sub(pattern, phrase + "。", text)
    return text


step4 = [remove_repetitive_phrases(t) for t in step3]
for t in step4:
    print(" -", t)

 - 您好！如需聯絡，請寄信到 [EMAIL] 或撥打 [PHONE]。您的信用卡卡號 [CREDIT_CARD] 已經過期。本訊息僅提供收件者參考，詳見 本站。
 - 頭條快訊 OpenAI 宣布推出新一代語言模型！ 版權所有，轉載請註明出處
 - 限時優惠不要錯過。立即購買 AI 課程！
 - GPT-3 擁有 1,750 億個參數，透過 Few-shot 就能完成新任務，詳情請見官方部落格。
 - 歡迎加入我們的 AI 讀書會，報名請寄 [EMAIL] 或私訊，銀行卡號 [CREDIT_CARD] 僅供測試。
 - 根據 GPT-3 論文的實驗結果，模型規模越大，Few-shot 表現通常也越好，但這不代表模型完全理解了任務本身。


---
## 清理前 vs 清理後

把整條管線串起來，對照 10 筆原始資料，最後活下來的只有幾篇。

In [7]:
print("=" * 60)
print(f"原始資料：{len(raw_dataset)} 筆")
print(f"最終保留：{len(step4)} 筆")
print("=" * 60)

for i, text in enumerate(step4, 1):
    print(f"\n【保留 {i}】{text}")

原始資料：10 筆
最終保留：6 筆

【保留 1】您好！如需聯絡，請寄信到 [EMAIL] 或撥打 [PHONE]。您的信用卡卡號 [CREDIT_CARD] 已經過期。本訊息僅提供收件者參考，詳見 本站。

【保留 2】頭條快訊 OpenAI 宣布推出新一代語言模型！ 版權所有，轉載請註明出處

【保留 3】限時優惠不要錯過。立即購買 AI 課程！

【保留 4】GPT-3 擁有 1,750 億個參數，透過 Few-shot 就能完成新任務，詳情請見官方部落格。

【保留 5】歡迎加入我們的 AI 讀書會，報名請寄 [EMAIL] 或私訊，銀行卡號 [CREDIT_CARD] 僅供測試。

【保留 6】根據 GPT-3 論文的實驗結果，模型規模越大，Few-shot 表現通常也越好，但這不代表模型完全理解了任務本身。


---
## 停下來想一想

- 語言過濾那一步，我們用的是「漢字比例」這種陽春規則，而不是真正的語言模型。如果今天資料裡出現大量「中英夾雜」的技術文章（這在 AI 領域的文章非常常見），這個規則還可靠嗎？
- MinHash 去重會留下「先出現的那一篇」，如果先出現的那篇品質其實比較差，這條規則就默默把品質差的內容留了下來。你會怎麼調整？
- day3.md 提到模型公司「買書、掃描、銷毀」的爭議新聞——這條清理管線示範的是「去重、去個資、去雜訊」，但完全沒有處理「這些內容有沒有取得授權」這一層問題。資料乾不乾淨，跟資料能不能用，其實是兩件不同的事。

這條管線只是縮小版，但背後的道理跟 GPT-2 清理 WebText、LLaMA 清理數十兆 token 的邏輯是一樣的：**規模變大之後，資料品質不會自動變好，反而更需要一條清楚、可重複執行的清理流程。**